# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



### Phase 0: Examine the Dataset (Meta)

SQL in sqlite3

In [2]:
import sqlite3 as db
import pandas as pd
import contextlib

In [3]:
%%capture
%%bash
apt-get update
apt-get install -y sqlite3

In [ ]:
! sqlite3 --help

In [ ]:
! sqlite3 --version

In [4]:
# Download the dataset
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip

Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [5]:
!ls -l

total 304
-rw-r--r-- 1 root root 305596 Nov 11 08:56 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


In [6]:
!rm -f chinook.db

In [7]:
!ls -l

total 304
-rw-r--r-- 1 root root 305596 Nov 11 08:56 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


In [8]:
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [9]:
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 11 08:56 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


In [10]:
# Get a list of the tables in the database
%%script sqlite3 --column --header chinook.db
.tables

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [11]:
# Show the schema for the entire database
%%script sqlite3 --column --header chinook.db
.schema

CREATE TABLE IF NOT EXISTS "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
);
CREATE TABLE sqlite_sequence(name,seq);
CREATE TABLE IF NOT EXISTS "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);
CREATE TABLE IF NOT EXISTS "customers"
(
    [CustomerId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    FOREIGN KEY ([SupportRepId]) REFERENCES "employees" ([EmployeeId]) 
		ON DELETE NO ACTION ON 

### Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



#### 1. Artist

In [12]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from artists
limit 10

ArtistId  Name                
--------  --------------------
1         AC/DC               
2         Accept              
3         Aerosmith           
4         Alanis Morissette   
5         Alice In Chains     
6         Antônio Carlos Jobim
7         Apocalyptica        
8         Audioslave          
9         BackBeat            
10        Billy Cobham        


In [28]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('artists') -- must have 'artists', not artists
PRAGMA table_info(artists);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    ArtistId  INTEGER        1                    1 
1    Name      NVARCHAR(120)  0                    0 


In [23]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('artists')
WHERE pk > 0;

name      type     pk
--------  -------  --
ArtistId  INTEGER  1 


In [30]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('artists');

In [14]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('artists')

total_columns
-------------
2            


In [18]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_artists
from artists

total_artists
-------------
275          


In [33]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM artists
WHERE Name IS NULL;

null_names
----------
0         


In [34]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL THEN 1 ELSE 0 END) AS null_artist_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM artists;

total_rows  null_artist_ids  null_names
----------  ---------------  ----------
275         0                0         


In [35]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL OR Name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM artists;

total_rows  rows_with_nulls
----------  ---------------
275         0              


In [36]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL THEN 1 ELSE 0 END) AS null_artist_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN ArtistId IS NULL OR Name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM artists;

total_rows  null_artist_ids  null_names  rows_with_nulls
----------  ---------------  ----------  ---------------
275         0                0           0              


In [37]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT Name) AS unique_artist_names
FROM artists;

unique_artist_names
-------------------
275                


In [38]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT Name, COUNT(*) AS occurrences
FROM artists
GROUP BY Name
ORDER BY Name;

Name                                                                                   occurrences
-------------------------------------------------------------------------------------  -----------
A Cor Do Som                                                                           1          
AC/DC                                                                                  1          
Aaron Copland & London Symphony Orchestra                                              1          
Aaron Goldberg                                                                         1          
Academy of St. Martin in the Fields & Sir Neville Marriner                             1          
Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner            1          
Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair  1          
Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart              1          
Academy of

In [39]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    ArtistId,
    Name,
    COUNT(*) AS occurrences
FROM artists
GROUP BY ArtistId, Name
HAVING COUNT(*) > 1;

In [40]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(ArtistId) FROM artists

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(ArtistId) FROM artists)
)

SELECT id AS missing_artist_id
FROM expected_ids
WHERE id NOT IN (
    SELECT ArtistId FROM artists
);

In [41]:

%%script sqlite3 --column --header chinook.db

-- =============================================
-- 1. TABLE STRUCTURE
-- =============================================

PRAGMA table_info(artists);


-- =============================================
-- 2. ROW AND COLUMN COUNTS
-- =============================================

SELECT
    COUNT(*) AS total_rows
FROM artists;


-- =============================================
-- 3. NULL COUNTS
-- =============================================

SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN ArtistId IS NULL THEN 1 ELSE 0 END)
        AS null_ArtistId,

    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END)
        AS null_Name,

    SUM(
        CASE
            WHEN ArtistId IS NULL OR Name IS NULL
            THEN 1 ELSE 0
        END
    ) AS rows_with_nulls

FROM artists;


-- =============================================
-- 4. CATEGORICAL ANALYSIS
-- =============================================

-- Count distinct categories
SELECT
    COUNT(DISTINCT Name) AS unique_artist_names
FROM artists;


-- List categories and their frequencies
SELECT
    Name,
    COUNT(*) AS occurrences
FROM artists
GROUP BY Name
ORDER BY Name;


-- =============================================
-- 5. DUPLICATE ANALYSIS
-- =============================================

-- Identify duplicate records
SELECT
    ArtistId,
    Name,
    COUNT(*) AS occurrences
FROM artists
GROUP BY ArtistId, Name
HAVING COUNT(*) > 1;


-- Count excess duplicate rows
SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM artists
    GROUP BY ArtistId, Name
    HAVING COUNT(*) > 1
);


-- =============================================
-- 6. MISSING IDENTIFIER ANALYSIS
-- =============================================

WITH RECURSIVE expected_ids(id) AS (

    SELECT MIN(ArtistId)
    FROM artists

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(ArtistId) FROM artists)

)

SELECT
    id AS missing_artist_id
FROM expected_ids
WHERE id NOT IN (
    SELECT ArtistId
    FROM artists
);


cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    ArtistId  INTEGER        1                    1 
1    Name      NVARCHAR(120)  0                    0 
total_rows
----------
275       
total_rows  null_ArtistId  null_Name  rows_with_nulls
----------  -------------  ---------  ---------------
275         0              0          0              
unique_artist_names
-------------------
275                
Name                                                                                   occurrences
-------------------------------------------------------------------------------------  -----------
A Cor Do Som                                                                           1          
AC/DC                                                                                  1          
Aaron Copland & London Symphony Orchestra                                              1          
Aaron Goldberg                    

### Phase 2. Map the Terrain (Get Familiar with the Data)



Data never exists in a vacuum. By writing queries, you will uncover the shape, structure, and quirks of the Chinook dataset. Again, using the SQL statements below, you'll learn how the tables relate to one another, what a single row represents in different contexts, and where specific types of information live.  Here's where you should get familiar with the ERD and join tables in various ways.



### Phase 3. Answer Guided Questions



Once you know the syntax and understand the data structure, you will put your skills to work solving specific business problems. Below are sets of explicit questions to answer using SQL. This phase mimics standard reporting: you are handed a specific prompt, and your job is to translate that prompt into an accurate, data-driven answer.

Here are some practice questions:
- https://github.com/LucasMcL/15-sql_queries_02-chinook/blob/master/README.md
- [Using the R language]( https://rpubs.com/enext777/636199 )
- [Search Google]( https://www.google.com/search?q=chinook+database+questions )
- Ask AI to generate some questions for you.  Be sure to mention the Chinook data set.




### Phase 4. Ask and Answer Your Own Questions


This final phase is the most critical part of the project. The true hallmark of a great data scientist isn't just answering the questions you're handed; it's having the curiosity and intuition to formulate your own questions.

You are tasked with looking at the Chinook dataset, generating your own original hypotheses or business questions, and writing the SQL required to answer them.

Demonstrate that you know SQL fundamentals by posting a question for each SQL statement below and then using the SQL statement to answer you question.

You will use the material from this phase for the presentation.



## Why This Matters: The Path to Your Capstone


Learning to ask your own questions is the exact skill you will rely on for your final capstone project. In your capstone, you will choose a dataset from your own field of interest. No one will hand you a list of prompts or a guide telling you what to look for.

Use this project as your sandbox. Mastering the ability to interrogate a database now will give you the framework you need to discover meaningful, independent insights in your capstone later.



## Chinook data set

See the lecture on SQLite3 using the Chinook data set to set up the software, database, and tables, as well as for the links to ancillary information about the data set.


In [ ]:
# Install the sqlite package for Ubuntu
# Download the Chinook sqlite database


## Use SQL queries to pull specific information

Do NOT pull all the data and then filter using DataFrame methods.  The initial purpose is to get familiar with SQL.




### Basic Queries


#### SELECT (with * and with column names)


#### WHERE


#### AND


#### OR


#### LIKE (with % or _ wildcard)


#### BETWEEN


#### LIMIT



### Sorting and Grouping


#### ORDER BY


#### DISTINCT


#### GROUP BY



### Aggregates


#### MAX


#### MIN


#### SUM


#### AVG


#### COUNT



## Make some plots

Make some cool plots to go with your data. Write SQL queries to get ONLY the information you need for each plot. (Don't pull ALL the data and then just plot a few columns.)



## EXTRA CREDIT:
* Use a CTE
* Use a query that joins two or more tables.
* Make a model to see if you can predict something ( e.g. GNB classifier, linear regression )
* Come up with something else cool to do with your data
